# 01 Fetch OpenNeuro Data & Preprocess

In [ ]:
from __future__ import annotations

import gzip
import json
import re
from dataclasses import asdict, dataclass
from pathlib import PurePosixPath
from typing import Optional

import boto3
from botocore import UNSIGNED
from botocore.config import Config

import pandas as pd

from segmentation.openneuro import (
    build_or_load_index,
    fetch_openneuro_files,
    fetch_build_stamp_cache,
    get_all_build_stamp_paths,
    get_first_build_stamp_path_per_dataset,
    make_build_stamp_fetch_records,
    extract_fs_versions,
    build_fs_file_map_from_cache,
    list_all_files,
    convert_file_map_to_tensors,
    prepare_images_if_needed,
)


In [ ]:
idx = build_or_load_index("openneuro_index.json.gz")
pairs = idx.pair_aparc_with_fs_inputs()

In [ ]:
# drop tiny datasets with less than 5 subjects
x = pd.Series([p.dataset_id for p in pairs]).value_counts()
x = x[x > 5]
ds_keep = x.index.values.tolist()
pairs_filt = [p for p in pairs if p.dataset_id in ds_keep]

df = pd.DataFrame(dict(
    orig_key=[p.preferred_orig_key for p in pairs_filt],
    rawavg_key=[p.preferred_rawavg_key for p in pairs_filt],
))

inds = ~df["rawavg_key"].isna()
pairs_filt = [p for i, p in enumerate(pairs_filt) if inds[i]]

In [90]:
report = fetch_openneuro_files(
    pairs_filt,
    out_root="openneuro_cache/files",
    which=("aparc_key", "preferred_rawavg_key"),
    max_workers=16,
    skip_existing=True
)

In [222]:
import numpy as np
import json
with open("fs_file_map.json", "r", encoding="utf-8") as f:
    file_map = json.load(f)

file_map_filt = {k: v for k, v in file_map.items()
     if file_map[k]["rawavg"] is not None and file_map[k]["aparc+aseg"] is not None}

x = [f for f in file_map.keys() if "openneuro_cache" in f]
ds = np.array([i.strip("/home/rph/convolutional_ar/segmentation/openneuro_cache/files").split("/")[0] for i in x])
ds = np.unique(ds).tolist()
len(ds)

52

In [ ]:
# 1) ALL build-stamp.txt paths for every dataset in your ds list
build_stamp_paths = get_all_build_stamp_paths(idx, dataset_ids=ds)

# 2) first build-stamp.txt path per dataset, preferring /sub- paths
first_build_stamp = get_first_build_stamp_path_per_dataset(idx, dataset_ids=ds)

# 3) sanity checks
missing_keys = [str(i) for i in ds if str(i) not in build_stamp_paths]   # should be []
no_build_stamp = [k for k, v in build_stamp_paths.items() if len(v) == 0]

records = make_build_stamp_fetch_records(idx, dataset_ids=ds)
# [r for r in records if r["dataset_id"] == "ds001734"][0]

In [ ]:
out = fetch_build_stamp_cache(
    first_build_stamp,
    cache_root="fs_build_stamp_cache/files",
    max_workers=16,
    skip_existing=True,
)

texts = out["text_by_dataset"]

version_by_dataset = extract_fs_versions(texts)

Fetching build-stamp.txt:   0%|          | 0/84 [00:00<?, ?it/s]

In [ ]:
ds = np.array([
    i.strip("/home/rph/convolutional_ar/segmentation/openneuro_cache/files").split("/")[0]
    for i in x
])
ds = np.unique(ds)
len(ds) # 52 studies

52

In [ ]:
for i in ds:
    assert i in texts.keys()

texts_matched = {k: v for k, v in texts.items() if k in ds}

version_by_dataset = extract_fs_versions(texts_matched)

# manual edits to ensure fs version exists - the code missed these, I manually added them from logs on openneuro
# ds005032: error 404
version_by_dataset["ds005234"] = "7.4.1"
version_by_dataset["ds007386"] = "5.1.0"
version_by_dataset["ds007387"] = "7.3.2"
version_by_dataset["ds007388"] = "7.3.2"
version_by_dataset["ds007391"] = "7.3.2"

# save
with open("version_by_dataset.json", "w", encoding="utf-8") as f:
    json.dump(version_by_dataset, f, indent=2)

## To Torch

In [ ]:
file_map = build_fs_file_map_from_cache("openneuro_cache")

In [2]:
files = list_all_files("hcp_filt")
files = [f for f in files if "T1w/" in f and
         (f.endswith("T1w_acpc_dc_restore.nii.gz") or f.endswith("aparc+aseg.nii.gz"))]

for f in files:
    s = f.split("/")
    k = "/".join(s[:-1])
    s = s[-1].split(".")[0]

    if k not in file_map.keys():
        file_map[k] = {"rawavg": None, "aparc+aseg": None}

    if s == "T1w_acpc_dc_restore":
        s = "rawavg"
    file_map[k][s] = f

In [3]:
file_map_filt = {
     k: v for k, v in file_map.items()
     if v["aparc+aseg"] is not None and v["rawavg"] is not None
}

len(file_map_filt), len(file_map)

(4264, 4393)

In [ ]:
results = convert_file_map_to_tensors(file_map_filt, out_root="tensors", n_jobs=-1)

Converting:   0%|          | 0/4264 [00:00<?, ?it/s]